# 🛒 Market Recommender System
## 🧩 Product Features

---

# 📌 Objetivo

Este notebook tem como objetivo construir e enriquecer as principais características dos produtos disponíveis no dataset Instacart Market Basket Analysis.

Como o projeto seguirá uma abordagem de recomendação **produto → produto**, o foco desta etapa será preparar informações úteis para identificar padrões de afinidade entre itens frequentemente comprados em conjunto.

As features construídas serão utilizadas nas próximas etapas para:

- analisar popularidade dos produtos;
- avaliar comportamento de recompra;
- enriquecer produtos com `aisle` e `department`;
- apoiar a construção de análises de coocorrência;
- preparar insumos para modelos de recomendação baseados em similaridade, embeddings ou MLP.

# 📚 Bibliotecas

Nesta seção são importadas as bibliotecas utilizadas para manipulação dos dados, configuração do ambiente do notebook e exportação dos arquivos processados.

In [ ]:
# =========================================
# 📚 IMPORTAÇÃO DAS BIBLIOTECAS
# =========================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# 📂 Carregamento dos Dados

Nesta etapa são carregadas as tabelas necessárias para a construção das features de produto.

Datasets utilizados:

- `products`
- `aisles`
- `departments`
- `order_products_prior`
- `order_products_train`
- `orders`

A tabela `order_products_prior` representa o histórico de compras e será utilizada para calcular métricas agregadas dos produtos.

In [ ]:
# =========================================
# 📂 CARREGAMENTO DOS DADOS
# =========================================

DATA_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(DATA_PATH / "orders.csv")
products = pd.read_csv(DATA_PATH / "products.csv")
aisles = pd.read_csv(DATA_PATH / "aisles.csv")
departments = pd.read_csv(DATA_PATH / "departments.csv")

order_products_prior = pd.read_csv(DATA_PATH / "order_products__prior.csv")
order_products_train = pd.read_csv(DATA_PATH / "order_products__train.csv")

# 🔍 Visão Geral da Base

Antes da criação das features, é importante validar o volume dos datasets e compreender a divisão dos pedidos entre `prior`, `train` e `test`.

Essa verificação ajuda a confirmar que o conjunto `prior` será usado como histórico de comportamento e que os demais conjuntos serão utilizados nas etapas posteriores de modelagem e avaliação.

In [ ]:
# =========================================
# 🔍 VISÃO GERAL DOS DATASETS
# =========================================

datasets = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

In [ ]:
# =========================================
# 📊 DISTRIBUIÇÃO DOS PEDIDOS
# =========================================

orders["eval_set"].value_counts()

💡 **Observação**:

A maior parte dos pedidos pertence ao conjunto `prior`, que representa o histórico de interações dos usuários com os produtos.

Para a abordagem produto → produto, esse histórico é especialmente importante, pois permite identificar quais itens aparecem juntos nos mesmos pedidos e quais produtos apresentam maior recorrência de consumo.

# 🛒 Enriquecimento dos Produtos

Nesta etapa, a tabela de produtos será enriquecida com informações de corredor (`aisle`) e departamento (`department`).

Esse enriquecimento melhora a interpretabilidade das recomendações, permitindo entender não apenas quais produtos são sugeridos, mas também a quais categorias eles pertencem.

In [ ]:
# =========================================
# 🛒 PRODUTOS ENRIQUECIDOS
# =========================================

products_enriched = (
    products
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

products_enriched.head()

# 📈 Popularidade dos Produtos

A popularidade indica quantas vezes cada produto foi comprado no histórico de pedidos.

Essa métrica é útil como baseline de recomendação e também pode ser combinada com técnicas de coocorrência e similaridade para priorizar itens mais relevantes.

In [ ]:
# =========================================
# 📈 TOTAL DE COMPRAS POR PRODUTO
# =========================================

product_purchase_count = (
    order_products_prior
    .groupby("product_id")
    .size()
    .reset_index(name="total_purchases")
    .sort_values("total_purchases", ascending=False)
)

product_purchase_count.head()

# 🔁 Taxa de Recompra por Produto

A taxa de recompra representa a proporção de vezes em que um produto foi comprado novamente pelos usuários.

Produtos com alta taxa de recompra tendem a representar itens de consumo recorrente, sendo importantes para estratégias de recomendação e reposição automática.

In [ ]:
# =========================================
# 🔁 TAXA DE RECOMPRA POR PRODUTO
# =========================================

product_reorder_rate = (
    order_products_prior
    .groupby("product_id")["reordered"]
    .mean()
    .reset_index(name="reorder_rate")
)

product_reorder_rate.head()

# 🧩 Dataset Final de Product Features

Nesta etapa, as informações de produto, popularidade, recompra, corredor e departamento são consolidadas em uma única tabela.

Esse dataset será utilizado como insumo para as próximas etapas do projeto, especialmente na análise de afinidade entre produtos e construção do mecanismo de recomendação.

In [ ]:
# =========================================
# 🧩 DATASET DE FEATURES DE PRODUTO
# =========================================

product_features = (
    products_enriched
    .merge(product_purchase_count, on="product_id", how="left")
    .merge(product_reorder_rate, on="product_id", how="left")
)

product_features[["total_purchases", "reorder_rate"]] = product_features[["total_purchases", "reorder_rate"]].fillna(0)

product_features.head()

# 🔍 Validação das Features

Antes de exportar os dados processados, serão realizadas verificações simples de consistência:

- dimensão da tabela final;
- presença de valores ausentes;
- tipos de dados;
- visualização dos produtos mais populares.

In [ ]:
# =========================================
# 🔍 VALIDAÇÃO DO DATASET FINAL
# =========================================

print("Shape:", product_features.shape)

product_features.info()

In [ ]:
# =========================================
# 🔍 VALORES AUSENTES
# =========================================

missing_values = (
    product_features
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

In [ ]:
# =========================================
# 🏆 TOP PRODUTOS MAIS POPULARES
# =========================================

product_features
    .sort_values("total_purchases", ascending=False)
    .head(10)[[
        "product_id",
        "product_name",
        "aisle",
        "department",
        "total_purchases",
        "reorder_rate",
    ]]

💡 **Observação**:

As features de produto representam uma camada importante para o sistema de recomendação, pois descrevem o comportamento geral dos itens dentro da plataforma.

A combinação entre popularidade, taxa de recompra, aisle e department permite enriquecer análises posteriores de coocorrência e similaridade, tornando as recomendações mais interpretáveis e alinhadas ao comportamento real de consumo.

# 💾 Exportação dos Dados Processados

Os datasets processados serão salvos em formato `.parquet`, garantindo leitura mais eficiente nas próximas etapas do projeto.

Arquivos gerados:

- `products_enriched.parquet`
- `product_features.parquet`

Esses arquivos poderão ser utilizados pelos notebooks de análise de cesta, similaridade entre produtos e treinamento de modelos de recomendação.

In [ ]:
# =========================================
# 💾 EXPORTAÇÃO DOS DATASETS PROCESSADOS
# =========================================

products_enriched.to_parquet(
    PROCESSED_PATH / "products_enriched.parquet",
    index=False,
)

product_features.to_parquet(
    PROCESSED_PATH / "product_features.parquet",
    index=False,
)

print("✅ Arquivos exportados com sucesso!")

# 📌 Conclusão do Product Features

Nesta etapa foram construídas e enriquecidas as principais características dos produtos disponíveis na plataforma.

As variáveis geradas representam aspectos importantes de popularidade, frequência de compra, comportamento de recompra e categorização dos produtos por corredores e departamentos.

Essas informações servirão como base para as próximas fases do projeto, focadas em:

- análise de cestas de compra;
- identificação de produtos frequentemente adquiridos em conjunto;
- cálculo de afinidade entre itens;
- criação de recomendações produto → produto;
- preparação de insumos para modelos MLP ou embedding-based em PyTorch.

Dessa forma, o notebook atua como uma etapa intermediária entre o EDA e os módulos de recomendação, mantendo o pipeline mais organizado, interpretável e alinhado ao objetivo final do projeto.